# Laya V4.6 — Marker-Aware Head Compression + Distillation + Exact Score-Only Inference

**Experiment:** Accelerate the original pretrained Laya model by eliminating unnecessary action-head execution and testing a smaller *typed-decision head input*, rather than approximating ModernBERT attention.

**Why:** V4.3 matched teacher decisions but ran at 0.625× speed; V4.4 bounded kernels were slower on T4; V4.5 rejected all 18 sliding-window replacements (selected=[]). Attention replacements should not be deployed. The original pretrained 28-layer ModernBERT encoder, Q/K/V, RoPE, residuals and FFNs remain intact.

**New:** (1) exact score-only path, (2) marker+CLS+landmark token budgets of 32/64/128/256 for the **two-layer post-encoder decision head only**, (3) optional head compilation; (4) quality and paired GPU/wall-latency gates; (5) single unseen AUDIT, no checkpoint selection on audit.

**Warning:** This notebook is a *testable hypothesis*, not a demonstrated speedup. Do not claim success until it passes GPU and held-out audit. Score-only mode intentionally omits the auxiliary action logits. Training-free compressed head may fail fidelity; the native teacher is the fallback.

[Original V4.3](https://colab.research.google.com/github/vtavakkoli/TinyCeNN-LM/blob/main/notebooks/Laya_V43_V31_Fidelity_First_Full_Then_Sliding_Colab.ipynb) · [V4.5 report explanation](https://github.com/vtavakkoli/TinyCeNN-LM/blob/main/notebooks/Laya_V45_Lean_Attention_Colab.ipynb)


In [ ]:
#@title 1. Dependencies and GPU
%pip -q install "transformers==4.57.6" "datasets>=2.20" "huggingface_hub>=0.25,<1" "safetensors>=0.4"
%pip -q install -U "git+https://github.com/vtavakkoli/TinyCeNN-LM.git@main"
import torch
assert torch.cuda.is_available(), "In Colab select Runtime > Change runtime type > GPU."
print("Torch",torch.__version__,"CUDA",torch.version.cuda,"GPU",torch.cuda.get_device_name(0))


In [ ]:
#@title 2. Reproducible source model and experiment configuration
from pathlib import Path
import math, json, random, time, copy, gc, statistics
import numpy as np
import torch
from torch import nn
from torch.nn import functional as F
from datasets import load_dataset
from huggingface_hub import snapshot_download
from transformers import AutoTokenizer
from tinycenn_lm.standalone_decision import (QTYPES, build_checkpoint_model,
    build_sequence, collate_items, normalize_question)

SOURCE_MODEL="convaiinnovations/laya-typed-decisions"
SEED=2046
MAX_LEN=512 #@param {type:"integer"}
TRAIN_CASES=500
DEV_CASES=240
BATCH=2 #@param {type:"integer"}
PROBE_DECISIONS=120 #@param {type:"integer"}
AUDIT_DECISIONS=200 #@param {type:"integer"}
TOKEN_BUDGETS=(32,64,128,256) # marker+CLS tokens are added
BENCH_REPEATS=10 #@param {type:"integer"}
MIN_TEACHER_AGREEMENT=0.99 #@param {type:"number"}
MAX_GOLD_DROP_PP=1.5 #@param {type:"number"}
MIN_SPEEDUP=1.03 #@param {type:"number"}
ENABLE_HEAD_COMPILE=True #@param {type:"boolean"}
ENABLE_FULL_COMPILE=False #@param {type:"boolean"}
SAVE_TO_GOOGLE_DRIVE=False #@param {type:"boolean"}
OUT=Path("/content/Laya_V46_Marker_Head")
if SAVE_TO_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    OUT=Path("/content/drive/MyDrive/TinyCeNN-LM/V46")
OUT.mkdir(parents=True,exist_ok=True)
assert 1<=BATCH<=8 and BENCH_REPEATS>=5 and 0<MIN_TEACHER_AGREEMENT<=1
assert MAX_GOLD_DROP_PP>=0 and MIN_SPEEDUP>=1.0
random.seed(SEED); np.random.seed(SEED);torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
device=torch.device("cuda")
amp_dtype=(torch.bfloat16 if torch.cuda.is_bf16_supported(including_emulation=False)
           else torch.float16)
source_dir=Path(snapshot_download(SOURCE_MODEL, allow_patterns=[
    "rl_agent_config.json","model.safetensors","encoder/*","tokenizer/*"]))
teacher,source_cfg=build_checkpoint_model(source_dir)
teacher.to(device).eval().requires_grad_(False)
teacher.encoder.config.reference_compile=False
tokenizer=AutoTokenizer.from_pretrained(str(source_dir/"tokenizer"))
assert teacher.head is not None, "V4.6 requires a pretrained decision Transformer head"
assert MAX_LEN <= int(source_cfg.get("max_len",MAX_LEN))
print("Original model parameters:",sum(p.numel() for p in teacher.parameters()))
print("Decision head layers:",len(teacher.head.layers),"AMP dtype",amp_dtype)


In [ ]:
#@title 3. Case-disjoint TRAIN / PROBE / AUDIT data + batch utilities
def _jsonish(x):
    if isinstance(x, str):
        try:
            return json.loads(x)
        except Exception:
            return x
    return x

def _canon_label(x):
    s = str(x).strip().lower()
    try:
        v = float(s)
        if math.isfinite(v) and v.is_integer():
            return str(int(v))
    except Exception:
        pass
    return s

def _gold_metadata(q, gold_answer):
    if not isinstance(gold_answer, dict):
        raise ValueError("missing gold answer")
    probs_raw = _jsonish(gold_answer.get("probabilities", {}))
    if not isinstance(probs_raw, dict) or len(probs_raw) < 2:
        raise ValueError("gold probabilities missing or malformed")

    gold_keys = [str(k) for k in probs_raw.keys()]
    if q["t"] == "choice":
        labels = [str(k) for k in q["crit"].keys()]
    elif q["t"] == "noul":
        labels = ["false", "true"]
    else:
        labels = list(gold_keys)
        if all(_canon_label(k).lstrip("-").isdigit() for k in labels):
            labels = sorted(labels, key=lambda k: int(_canon_label(k)))

    lookup = {_canon_label(k): float(v) for k, v in probs_raw.items()}
    if len(labels) != len(gold_keys) or any(_canon_label(k) not in lookup for k in labels):
        raise ValueError("Gold labels do not match encoded option order")

    gold_probs = np.asarray([lookup[_canon_label(k)] for k in labels], dtype=np.float64)
    gold_probs = np.clip(gold_probs, 0.0, None)
    if not np.isfinite(gold_probs).all() or gold_probs.sum() <= 0:
        raise ValueError("invalid gold probability vector")
    gold_probs /= gold_probs.sum()

    gold_label = _canon_label(gold_answer.get("label", labels[int(gold_probs.argmax())]))
    canon_labels = [_canon_label(k) for k in labels]
    if gold_label not in canon_labels:
        raise ValueError(f"gold label {gold_label!r} not in {labels!r}")
    return labels, gold_probs.tolist(), canon_labels.index(gold_label)

def make_items_from_rows(rows, seed, shuffle_items):
    out = []
    head_max_len = int(source_cfg.get("head_max_len", 192))
    for row in rows:
        state = _jsonish(row["state"])
        questions = _jsonish(row["questions"])
        gold = _jsonish(row["gold"])
        case_id = str(row.get("id", ""))
        workflow = str(row.get("workflow", "unknown"))
        state_text = state if isinstance(state, str) else json.dumps(state, ensure_ascii=False)
        state_ids = tokenizer(
            state_text.replace(tokenizer.mask_token, " "),
            add_special_tokens=False,
        )["input_ids"]

        for qid, qdef in questions.items():
            q = normalize_question(qdef)
            ids, markers = build_sequence(
                tokenizer,
                state,
                q,
                max_len=min(MAX_LEN, int(source_cfg.get("max_len", MAX_LEN))),
                head_max_len=head_max_len,
                truncate_left=isinstance(state, list),
                state_ids=state_ids,
            )
            if len(markers) < 2:
                continue
            labels, gold_probs, gold_index = _gold_metadata(q, gold[qid])
            if len(labels) != len(markers):
                raise ValueError(
                    f"{case_id}/{qid}: {len(markers)} markers vs {len(labels)} gold labels"
                )
            out.append({
                "ids": ids,
                "markers": markers,
                "qtype": QTYPES[q["t"]],
                "qtype_name": q["t"],
                "qid": str(qid),
                "case_id": case_id,
                "workflow": workflow,
                "labels": labels,
                "gold_probs": gold_probs,
                "gold_index": int(gold_index),
            })
    if shuffle_items:
        random.Random(seed).shuffle(out)
    return out

rows = list(load_dataset("LocalLLaMA/typed-decisions", "all", split="train"))
rng = random.Random(SEED)
rng.shuffle(rows)
assert TRAIN_CASES + DEV_CASES <= len(rows)
train_rows = rows[:TRAIN_CASES]
dev_rows = rows[TRAIN_CASES:TRAIN_CASES + DEV_CASES]
train_items = make_items_from_rows(train_rows, SEED, True)
# Split at the CASE level before creating multiple question items per case.
probe_rows = dev_rows[:len(dev_rows) // 2]
audit_rows = dev_rows[len(dev_rows) // 2:]
probe_items = make_items_from_rows(probe_rows, SEED + 1, True)
audit_items = make_items_from_rows(audit_rows, SEED + 2, True)
dev_items = probe_items # backward-compatible alias for training helper use
assert train_items and probe_items and audit_items
assert {str(r["id"]) for r in probe_rows}.isdisjoint({str(r["id"]) for r in audit_rows})

def direct_batch(selected):
    b = collate_items(selected, int(tokenizer.pad_token_id))
    return {k: v.to(device) for k, v in b.items()}

def batch(items, offset, n=BATCH):
    selected = [items[(offset + i) % len(items)] for i in range(n)]
    return direct_batch(selected)

assert len(probe_items)>=PROBE_DECISIONS and len(audit_items)>=AUDIT_DECISIONS
assert {x["case_id"] for x in probe_items}.isdisjoint({x["case_id"] for x in audit_items})
print("TRAIN/PROBE/AUDIT items:",len(train_items),len(probe_items),len(audit_items))


In [ ]:
#@title 4. Architecture: native ModernBERT + marker-preserving decision head
# This model shares the original FROZEN pretrained weights. NO duplicated encoder,
# no CeNN branch, no learned new parameters, no training.
class MarkerAwareScores(nn.Module):
    def __init__(self, pretrained, budget=0):
        super().__init__()
        self.encoder=pretrained.encoder
        self.type_emb=pretrained.type_emb
        self.head=pretrained.head
        self.scorer=pretrained.scorer
        self.budget=int(budget)
        assert self.budget>=0

    def _select_head_tokens(self,h,attention_mask,marker_pos,marker_mask):
        # Keep CLS for the original typed-decision semantics; preserve *all*
        # option marker positions, then sample a fixed number of context tokens.
        # Duplicate markers and padded context are masked rather than attended.
        b,t,d=h.shape
        cls=torch.zeros((b,1),dtype=torch.long,device=h.device)
        lm_count=min(self.budget,max(t-1,1))
        lm=torch.linspace(1,max(t-1,1),steps=lm_count,device=h.device).round().long()
        lm=lm.clamp(max=t-1).view(1,-1).expand(b,-1)
        marker_ok=marker_mask.bool()
        lm_ok=attention_mask.bool().gather(1,lm)
        lm_ok=lm_ok & (lm!=0)
        # Markers may be padded to 0; their mask distinguishes real positions.
        duplicate=(lm[:,:,None]==marker_pos[:,None,:]) & marker_ok[:,None,:]
        lm_ok=lm_ok & ~duplicate.any(-1)
        idx=torch.cat((cls,marker_pos.clamp(0,t-1),lm),dim=1)
        keep=torch.cat((torch.ones((b,1),dtype=torch.bool,device=h.device),
                        marker_ok,lm_ok),dim=1)
        gathered=h.gather(1,idx[:,:,None].expand(-1,-1,d))
        return gathered,keep

    def forward(self,input_ids,attention_mask,marker_pos,marker_mask,qtype):
        h=self.encoder(input_ids=input_ids,
                       attention_mask=attention_mask).last_hidden_state
        h=h+self.type_emb(qtype)[:,None,:]
        if self.budget:
            compact,keep=self._select_head_tokens(h,attention_mask,
                                                    marker_pos,marker_mask)
            transformed=self.head(compact,src_key_padding_mask=~keep)
            marker_hidden=transformed[:,1:1+marker_pos.shape[1],:]
        else:
            transformed=self.head(h,src_key_padding_mask=~attention_mask.bool())
            marker_hidden=transformed.gather(1,marker_pos.clamp_min(0)[:,:,None]
                                             .expand(-1,-1,h.shape[-1]))
        logits=self.scorer(marker_hidden).squeeze(-1).float()
        logits=logits.masked_fill(~marker_mask,-1e4)
        # The original action logits are deliberately NOT computed here:
        # this variant serves option-score decisions only.
        return logits,None

exact_score_model=MarkerAwareScores(teacher,budget=0).eval()
assert all(not p.requires_grad for p in exact_score_model.parameters())
print("Ready: budget=0 reproduces the original option logits, omitting only act_logits")


In [ ]:
#@title 5. Strict quality tests and A/B speed measurement
@torch.inference_mode()
def probs_for(model,b):
    with torch.autocast("cuda",dtype=amp_dtype):
        scores,_=model(**b)
    return torch.softmax(scores.float().masked_fill(
        ~b["marker_mask"].bool(),-1e9),dim=-1).cpu()

@torch.inference_mode()
def get_reference(items,limit):
    out=[]
    for start in range(0,min(limit,len(items)),BATCH):
        chosen=items[start:min(start+BATCH,limit)]
        p=probs_for(teacher,direct_batch(chosen))
        for j,item in enumerate(chosen):
            out.append(dict(probs=p[j],teacher=int(p[j].argmax()),
                gold=int(item["gold_index"]),case=item["case_id"],qid=item["qid"]))
    return out

@torch.inference_mode()
def quality(model,items,ref):
    agree=correct=tcorrect=0;js=0.
    n=len(ref)
    for start in range(0,n,BATCH):
        chosen=items[start:min(n,start+BATCH)]
        p=probs_for(model,direct_batch(chosen))
        for j,item in enumerate(chosen):
            t=ref[start+j]["probs"]; s=p[j]
            assert s.shape==t.shape
            m=(s+t)/2
            js+=float(.5*((s*(s.clamp_min(1e-8).log()-m.clamp_min(1e-8).log())).sum()+
                            (t*(t.clamp_min(1e-8).log()-m.clamp_min(1e-8).log())).sum()))
            guess=int(s.argmax())
            agree+=int(guess==ref[start+j]["teacher"])
            correct+=int(guess==item["gold_index"])
            tcorrect+=int(ref[start+j]["teacher"]==item["gold_index"])
    return dict(decisions=n,teacher_agreement=agree/n,gold_accuracy=correct/n,
        teacher_gold_accuracy=tcorrect/n,gold_drop=(tcorrect-correct)/n,mean_js=js/n)

BENCH_BATCHES=[batch(probe_items,0,BATCH),batch(probe_items,11,BATCH)]
@torch.inference_mode()
def benchmark_paired(reference,candidate,repeats=BENCH_REPEATS):
    """Alternating A/B/B/A with CUDA events AND synchronized wall-clock timings.
    All timings exclude Colab compilation, dataset loading and model downloading."""
    results=[]
    for j,b in enumerate(BENCH_BATCHES):
        def once(m):
            torch.cuda.synchronize()
            a=torch.cuda.Event(enable_timing=True)
            z=torch.cuda.Event(enable_timing=True)
            before=time.perf_counter()
            a.record()
            with torch.autocast("cuda",dtype=amp_dtype):
                outputs=m(**b)
            z.record();z.synchronize()
            wall=(time.perf_counter()-before)*1000
            return float(a.elapsed_time(z)),float(wall)
        for k in range(4):
            once(reference);once(candidate)
        d={"reference":[],"candidate":[]}
        w={"reference":[],"candidate":[]}
        for k in range(repeats):
            order=("reference","candidate","candidate","reference") if k%2==0 else (
                "candidate","reference","reference","candidate")
            for name in order:
                gpu,cpu=once(reference if name=="reference" else candidate)
                d[name].append(gpu);w[name].append(cpu)
        ref=float(np.median(d["reference"]))
        cand=float(np.median(d["candidate"]))
        rw=float(np.median(w["reference"]))
        cw=float(np.median(w["candidate"]))
        results.append(dict(shape=j,batch=int(b["input_ids"].shape[0]),
            tokens=int(b["input_ids"].shape[1]),
            native_gpu_ms=ref,candidate_gpu_ms=cand,gpu_speedup=ref/max(cand,1e-9),
            native_wall_ms=rw,candidate_wall_ms=cw,wall_speedup=rw/max(cw,1e-9),
            native_p95_ms=float(np.percentile(d["reference"],95)),
            candidate_p95_ms=float(np.percentile(d["candidate"],95)),n=len(d["reference"])))
    return dict(median_gpu_speedup=float(np.median([r["gpu_speedup"] for r in results])),
        min_gpu_speedup=float(min(r["gpu_speedup"] for r in results)),
        median_wall_speedup=float(np.median([r["wall_speedup"] for r in results])),
        min_wall_speedup=float(min(r["wall_speedup"] for r in results)),
        per_shape=results)

def valid_quality(q):
    return (q["teacher_agreement"]>=MIN_TEACHER_AGREEMENT and
            q["gold_drop"]<=MAX_GOLD_DROP_PP/100 and q["mean_js"]<=0.003)
def valid_speed(s):
    return (s["min_gpu_speedup"]>=MIN_SPEEDUP and
            s["min_wall_speedup"]>=MIN_SPEEDUP)

PROBE_REF=get_reference(probe_items,PROBE_DECISIONS)
exact_q=quality(exact_score_model,probe_items,PROBE_REF)
print("Exact score-only PROBE:",exact_q)
assert exact_q["teacher_agreement"]==1.0 and abs(exact_q["mean_js"])<1e-7, (
    "Score-only implementation drifted from native pretrained teacher")
# Mandatory exact-output smoke test on a real, padded PROBE batch.
with torch.inference_mode(), torch.autocast("cuda",dtype=amp_dtype):
    _smoke=batch(probe_items,0,BATCH)
    _original_scores,_=teacher(**_smoke)
    _fast_scores,_=exact_score_model(**_smoke)
    _max_diff=float((_original_scores.float()-_fast_scores.float()).abs().max())
    assert torch.allclose(_original_scores.float(),_fast_scores.float(),
                          rtol=1e-5,atol=1e-5),_max_diff
print("Exact option-logit parity checked; max absolute error:",_max_diff)
print("Native benchmark shapes:",[(int(b["input_ids"].shape[0]),
                                  int(b["input_ids"].shape[1])) for b in BENCH_BATCHES])


In [ ]:
#@title 6. PROBE-only architecture selection (AUDIT untouched)
# Score-only mode is always a quality-parity control.
candidates=[(0,exact_score_model)]+[
    (k,MarkerAwareScores(teacher,budget=k).eval()) for k in TOKEN_BUDGETS]
records=[]
winning=None
winning_model=None
best_speed=1.0
for budget,model in candidates:
    try:
        q=quality(model,probe_items,PROBE_REF)
        passed_quality=valid_quality(q)
        # Avoid expensive benchmarking for architectures already too inaccurate.
        perf=benchmark_paired(teacher,model) if passed_quality else None
        passed_speed=(perf is not None and valid_speed(perf))
        accepted=passed_quality and passed_speed
        score=perf["median_gpu_speedup"] if perf is not None else 0.
        if accepted and score>best_speed:
            best_speed=score;winning=budget;winning_model=model
        rec=dict(budget=budget,quality=q,speed=perf,
                 quality_pass=passed_quality,speed_pass=passed_speed,
                 selected=False)
    except Exception as exc:
        rec=dict(budget=budget,error=repr(exc),quality_pass=False,speed_pass=False,
                 selected=False)
        print("Candidate failed safely:",budget,repr(exc))
    records.append(rec)
    print("budget",budget,"accuracy",rec.get("quality",{}).get("teacher_agreement"),
          "GPU speed",None if rec.get("speed") is None else round(score,3),
          "accepted",accepted if "error" not in rec else False)
    (OUT/"v46_probe_sweep.json").write_text(json.dumps(records,indent=2))
# No architecture speedup? Retain exact score-only candidate for runtime experiments,
# but do NOT treat it as selected/deployable until final comparison passes.
if winning_model is None:
    winning_model=exact_score_model
    winning=0
print("Provisional candidate budget:",winning,"passing strict speed gate:",best_speed>1)


In [ ]:
#@title 7. OPTIONAL: short teacher distillation to rescue token compression
# Only pretrained post-encoder decision-head weights are trainable; 28-layer
# ModernBERT and original teacher/score-only head remain frozen.
# Selection uses PROBE, never AUDIT. No new CeNN corrections or routing branches.
ENABLE_HEAD_DISTILL=True #@param {type:"boolean"}
DISTILL_BUDGETS=(128,256)
DISTILL_STEPS=60 #@param {type:"integer"}
DISTILL_LR=0.00003 #@param {type:"number"}
DISTILL_CHECK_EVERY=20 #@param {type:"integer"}
assert DISTILL_STEPS>=0 and DISTILL_CHECK_EVERY>=1
distill_records=[]

if ENABLE_HEAD_DISTILL and DISTILL_STEPS:
    for budget in DISTILL_BUDGETS:
        try:
            model=MarkerAwareScores(teacher,budget=budget).eval()
            model.head=copy.deepcopy(teacher.head).to(device)
            model.head.requires_grad_(True)
            optimizer=torch.optim.AdamW(model.head.parameters(),lr=DISTILL_LR,weight_decay=0.0)
            scaler=torch.amp.GradScaler("cuda",enabled=amp_dtype==torch.float16)
            best_snapshot=None
            best_probe=None
            best_candidate_speed=None
            # Hold original teacher and encoder frozen for all steps.
            teacher.eval()
            model.encoder.eval();model.type_emb.eval();model.scorer.eval()
            for step in range(1,DISTILL_STEPS+1):
                model.head.train()
                b=batch(train_items,(step*BATCH)%len(train_items),BATCH)
                with torch.no_grad(),torch.autocast("cuda",dtype=amp_dtype):
                    teacher_logits,_=teacher(**b)
                optimizer.zero_grad(set_to_none=True)
                with torch.autocast("cuda",dtype=amp_dtype):
                    student_logits,_=model(**b)
                    temperature=2.0
                    target=torch.softmax(teacher_logits.float()/temperature,dim=-1)
                    output_log=torch.log_softmax(student_logits.float()/temperature,dim=-1)
                    loss=F.kl_div(output_log,target,reduction="batchmean")*(temperature**2)
                if not torch.isfinite(loss):
                    raise FloatingPointError("Distillation loss not finite")
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.head.parameters(),1.0)
                scaler.step(optimizer);scaler.update()
                if step%DISTILL_CHECK_EVERY==0 or step==DISTILL_STEPS:
                    model.head.eval()
                    q=quality(model,probe_items,PROBE_REF)
                    if valid_quality(q):
                        perf=benchmark_paired(teacher,model,repeats=BENCH_REPEATS)
                        if valid_speed(perf):
                            speed=perf["median_gpu_speedup"]
                            if speed>best_speed:
                                best_speed=speed
                                winning=budget
                                winning_model=model
                                best_probe=q
                                best_candidate_speed=perf
                                best_snapshot={k:v.detach().cpu().clone()
                                               for k,v in model.head.state_dict().items()}
                    print("distill budget",budget,"step",step,
                          "loss",round(float(loss.item()),5),
                          "PROBE agreement",round(q["teacher_agreement"],4))
            # If this budget won, keep its frozen best checkpoint rather than
            # the possibly inferior last training step.
            if best_snapshot is not None and winning_model is model:
                model.head.load_state_dict(best_snapshot)
                model.head.eval().requires_grad_(False)
                head_path=OUT/("v46_distilled_head_budget_"+str(budget)+".pt")
                torch.save(dict(budget=budget,head_state_dict=best_snapshot,
                                source_model=SOURCE_MODEL),head_path)
                print("Selected train-only head checkpoint",head_path)
            distill_records.append(dict(budget=budget,steps=DISTILL_STEPS,
                                        selected=best_snapshot is not None and winning_model is model,
                                        probe=best_probe,performance=best_candidate_speed))
        except Exception as exc:
            distill_records.append(dict(budget=budget,error=repr(exc)))
            print("Distillation candidate failed safely:",budget,repr(exc))
        (OUT/"v46_distill_probe.json").write_text(json.dumps(distill_records,indent=2))
print("Architecture winner after training:",winning,"best speed on PROBE",best_speed)


In [ ]:
#@title 7. Optional torch.compile acceleration (PROBE only, safe fallback)
# Compile *decision head* first: less memory and much less compilation time
# than attempting to compile the entire 421M-parameter model on Tesla T4.
runtime_candidates=[("eager",winning_model)]
if ENABLE_HEAD_COMPILE:
    try:
        cm=MarkerAwareScores(teacher,budget=winning).eval()
        cm.head=torch.compile(winning_model.head,mode="reduce-overhead",dynamic=False)
        # Trigger compilation for the two benchmark shapes.
        with torch.inference_mode():
            for b in BENCH_BATCHES:
                for _ in range(3):
                    with torch.autocast("cuda",dtype=amp_dtype): cm(**b)
            torch.cuda.synchronize()
        runtime_candidates.append(("compile_head",cm))
        print("Compiled candidate's decision head.")
    except Exception as exc:
        print("Head compilation unavailable, using eager fallback:",repr(exc))
        torch.cuda.empty_cache()
if ENABLE_FULL_COMPILE:
    try:
        cc=torch.compile(winning_model,mode="reduce-overhead",dynamic=False)
        with torch.inference_mode():
            for b in BENCH_BATCHES:
                for _ in range(3):
                    with torch.autocast("cuda",dtype=amp_dtype): cc(**b)
            torch.cuda.synchronize()
        runtime_candidates.append(("compile_full",cc))
        print("Full model compiled.")
    except Exception as exc:
        print("Full compilation unavailable, using noncompiled fallback:",repr(exc))
        torch.cuda.empty_cache()

runtime_records=[]
final_choice=None
final_model=None
fastest=1.0
for name,model in runtime_candidates:
    try:
        q=quality(model,probe_items,PROBE_REF)
        s=benchmark_paired(teacher,model) if valid_quality(q) else None
        passed=valid_quality(q) and s is not None and valid_speed(s)
        rr=dict(name=name,budget=winning,quality=q,speed=s,accepted=passed)
        if passed and s["median_gpu_speedup"]>fastest:
            fastest=s["median_gpu_speedup"]
            final_choice=name;final_model=model
    except Exception as exc:
        rr=dict(name=name,budget=winning,accepted=False,error=repr(exc))
        print("Runtime path not accepted:",name,repr(exc))
    runtime_records.append(rr)
    print("Runtime",name,"accepted",rr["accepted"])
    (OUT/"v46_runtime_probe.json").write_text(json.dumps(runtime_records,indent=2))

if final_model is None:
    # No convincing acceleration, preserve the original native reference.
    final_choice="original_native"
    final_model=teacher
    chosen_budget=None
else:
    chosen_budget=winning
print("FROZEN selection:",final_choice,"budget:",chosen_budget)


In [ ]:
#@title 8. ONE untouched AUDIT, final latency, and reproducible export
# AUDIT must be evaluated exactly once after selection is FROZEN.
assert final_choice and final_model is not None
ref_audit=get_reference(audit_items,AUDIT_DECISIONS)
assert {r["case"] for r in ref_audit}.isdisjoint(
       {r["case"] for r in PROBE_REF})
final_audit=quality(final_model,audit_items,ref_audit)
final_perf=benchmark_paired(teacher,final_model,repeats=max(15,BENCH_REPEATS))
gates=dict(teacher_agreement=final_audit["teacher_agreement"]>=MIN_TEACHER_AGREEMENT,
    gold_drop=final_audit["gold_drop"]<=MAX_GOLD_DROP_PP/100,
    faster_than_native=(final_choice!="original_native" and valid_speed(final_perf)),
    non_native_execution=(final_choice!="original_native"))
gates["all_pass"]=all(gates.values())
result=dict(version="V4.6",source_model=SOURCE_MODEL,
    experiment="Marker-aware compressed decision-head vs native model",
    seed=SEED,device=torch.cuda.get_device_name(0),pytorch=torch.__version__,
    amp_dtype=str(amp_dtype),selected_runtime=final_choice,
    selected_context_budget=chosen_budget,probe_sweep=records,
    runtime_probe=runtime_records,distillation_probe=distill_records,final_audit=final_audit,
    final_paired_cuda=final_perf,gates=gates,
    warning="Only option-score predictions are returned by optimized runners; action logits omitted.")
filename=OUT/"v46_final_report.json"
filename.write_text(json.dumps(result,indent=2),encoding="utf-8")
print("AUDIT",final_audit)
print("Full-model GPU speedup",final_perf["median_gpu_speedup"])
print("Full-model synchronized wall-clock speedup",final_perf["median_wall_speedup"])
print("All gates",gates)
if not gates["all_pass"]:
    print("NOT VALIDATED FASTER: keep original pretrained checkpoint as deployment reference.")
else:
    print("PROVISIONAL SPEED + FIDELITY PASS on one T4 / small test; expand independent tests.")
print("Report:",filename)
from google.colab import files
files.download(str(filename))
